In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"

In [ ]:
try:
    engine = get_engine()
    df_races = load_races(engine, "2018-01-01", "2024-12-31")
    print(f"DB: {df_races['race_id'].nunique()} races, {len(df_races)} entries")
except Exception as e:
    print(f"DB接続失敗: {e} — mock データを使用")
    df_races = generate_mock_race_df(500)

In [ ]:
yearly = df_races.groupby(df_races["race_date"].dt.year).agg(
    n_races=("race_id", "nunique"),
    n_entries=("race_id", "count"),
    avg_odds=("win_odds", "mean"),
    median_odds=("win_odds", "median"),
)
display(yearly)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_races["win_odds"], bins=100, log=True, alpha=0.7, color="steelblue")
axes[0].set_xlabel("Win Odds")
axes[0].set_ylabel("Count (log)")
axes[0].set_title("Win Odds Distribution")
popularity_win_rate = df_races.groupby("popularity_rank").apply(
    lambda x: (x["finish_pos"] == 1).mean()
)
popularity_win_rate.head(10).plot.bar(ax=axes[1], color="coral")
axes[1].set_ylabel("Win Rate")
axes[1].set_title("Win Rate by Popularity Rank")
plt.tight_layout()
plt.savefig("01_eda_basic_stats.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for surface in ["turf", "dirt"]:
    sub = df_races[df_races["surface"] == surface]
    if len(sub) == 0:
        continue
    print(f"\n{surface}: {sub['race_id'].nunique()} races")
    print(f"  avg field size: {sub.groupby('race_id')['umaban'].count().mean():.1f}")
    fav_wr = (sub[sub['popularity_rank']==1]['finish_pos']==1).mean()
    print(f"  1st fav win rate: {fav_wr:.3f}")